# ARTI308 – Lab 4: Data Quality Assessment & Preprocessing  
**Dataset:** Adult Income (UCI) – `adult.data`  

This notebook completes the required tasks:  
1. Identify data quality issues  
2. Apply one missing value strategy and explain why  
3. Detect & handle outliers using IQR  
4. Normalize numerical features using **Min-Max** and **Z-score**  
5. Apply **PCA** and interpret explained variance  


In [ ]:
# 0) Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA

plt.rcParams["figure.figsize"] = (8, 4)


In [ ]:
# 1) Load the dataset (Adult Income)
columns = [
    "age","workclass","fnlwgt","education","education-num",
    "marital-status","occupation","relationship","race","sex",
    "capital-gain","capital-loss","hours-per-week",
    "native-country","income"
]

# If you're on Colab, upload adult.data to the left-side Files panel first.
df_raw = pd.read_csv("adult.data", names=columns, sep=",", skipinitialspace=True)

df_raw.head()


## Task 1 — Identify data quality issues  
We check:  
- Dataset shape (rows/columns)  
- Data types  
- Missing values (including `?`)  
- Duplicate rows  


In [ ]:
# Basic info
print("Shape (rows, columns):", df_raw.shape)
df_raw.info()


In [ ]:
# Missing values (true NaNs)
df_raw.isnull().sum().sort_values(ascending=False).head(20)


In [ ]:
# The Adult dataset uses '?' for unknown values. Count them.
question_counts = (df_raw == "?").sum().sort_values(ascending=False)
question_counts[question_counts > 0]


In [ ]:
# Duplicate rows
df_raw.duplicated().sum()


## Task 2 — Apply ONE missing value strategy (and explain why)  
Steps:  
1) Replace `?` with `NaN`  
2) Fill missing values using **Mode** (most frequent value) because missing columns are mainly categorical.  


In [ ]:
df = df_raw.copy()

# Replace '?' with NaN
df.replace("?", np.nan, inplace=True)

# Show missing values per column (before filling)
missing_before = df.isnull().sum().sort_values(ascending=False)
missing_before[missing_before > 0]


In [ ]:
# Fill missing values with Mode for each column that has missing values
for col in df.columns:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode(dropna=True)[0])

# Verify missing values are handled
df.isnull().sum().sum()


**Why Mode?**  
Mode is suitable here because most missing values occur in categorical variables such as `workclass`, `occupation`, and `native-country`. Filling with the most frequent category preserves common patterns without creating new categories.


## Task 3 — Detect & handle outliers using IQR  
We apply IQR to numerical columns and **cap** outliers to the lower/upper bounds (winsorization).  
This “handles” outliers while keeping the dataset size stable.


In [ ]:
numeric_cols = ["age","fnlwgt","education-num","capital-gain","capital-loss","hours-per-week"]

def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return lower, upper

# Count outliers per numeric column (before handling)
outlier_counts = {}
for col in numeric_cols:
    lo, hi = iqr_bounds(df[col])
    outlier_counts[col] = int(((df[col] < lo) | (df[col] > hi)).sum())

pd.Series(outlier_counts).sort_values(ascending=False)


In [ ]:
# Handle outliers by capping values to IQR bounds
df_out = df.copy()

for col in numeric_cols:
    lo, hi = iqr_bounds(df_out[col])
    df_out[col] = df_out[col].clip(lower=lo, upper=hi)

# Re-count outliers (should be 0 after clipping)
outlier_counts_after = {}
for col in numeric_cols:
    lo, hi = iqr_bounds(df_out[col])
    outlier_counts_after[col] = int(((df_out[col] < lo) | (df_out[col] > hi)).sum())

pd.Series(outlier_counts_after)


## Task 4 — Normalize numerical features  
We create TWO normalized versions:  
- `df_minmax` using **Min-Max scaling**  
- `df_zscore` using **Z-score standardization**


In [ ]:
# Min-Max Scaling
df_minmax = df_out.copy()

minmax = MinMaxScaler()
df_minmax[numeric_cols] = minmax.fit_transform(df_minmax[numeric_cols])

df_minmax[numeric_cols].head()


In [ ]:
# Z-score Standardization
df_zscore = df_out.copy()

zscaler = StandardScaler()
df_zscore[numeric_cols] = zscaler.fit_transform(df_zscore[numeric_cols])

df_zscore[numeric_cols].head()


## Task 5 — PCA and explained variance  
PCA is applied on standardized numeric data, so we use `df_zscore`.


In [ ]:
X = df_zscore[numeric_cols].values

pca = PCA()
pca.fit(X)

explained = pca.explained_variance_ratio_
cum_explained = np.cumsum(explained)

print("Explained variance ratio per component:")
print(np.round(explained, 4))

print("\nCumulative explained variance:")
print(np.round(cum_explained, 4))


In [ ]:
# Plot cumulative explained variance
plt.figure()
plt.plot(range(1, len(cum_explained)+1), cum_explained, marker="o")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA – Cumulative Explained Variance")
plt.ylim(0, 1.05)
plt.grid(True)
plt.show()


**Interpretation:**  
If the cumulative curve reaches a high value (e.g., 0.90) at component **k**, then the first **k** principal components preserve about **90%** of the variance (information) in the numeric features, while reducing dimensionality.


## Conclusion  
In this lab, we identified data quality issues, handled missing values using a mode strategy, detected and handled outliers using IQR, normalized numeric features using both Min-Max and Z-score, and applied PCA to analyze explained variance.
